In [1]:
import numpy as np
from numpy.random import randn
import random

In [29]:
train_data = {
    'хороший': True,
    'круто': True,
    'весело': True,
    'ахуенно': True,
    'ултра хорош': True,
    'супер мега хорош': True,
    'неплохо': True,
    'не грустно': True,
    'очень хорошо': True,
    'счаслив': True,
    'я хорош': True,
    'я охуителен': True,
    'мне хорошо': True,
    'мне не плохо': True,
    'Это очень хорошо': True,
    'хуево': False,
    'насрал': False,
    'плохо': False,
    'не хорошо': False,
    'кринжанул': False,
    'ультра насрал': False,
    'не радует': False,
    'радует': True,
    'грустно': False,
    'весело': True,
}

test_data = {
  'это счастье': True,
  'я хороший': True,
  'я не хороший': False,
  'мне не грустно': True,
  'я очень хорош': True,
  'мне очень грустно': False,
  'это плохо, а не хорошо': False,
}

Дальше не простой RNN и не библиотечный модуль. Два LSTM, `lstm1` и `lstm2`.
Каждый вентиль это отдельный слой нейронов: свои веса от входа и от скрытого состояния.
Вентили: забывание, вход, память, выход.

Позитив учится на фразах с `True`, негатив на фразах с `False`. Ниже к исходным фразам добавлен больший набор.


In [10]:
import re

def tokens(text):
    return re.findall(r"[a-zа-я0-9]+", text.lower().replace("ё", "е"))

def sigmoid(value):
    return 1 / (1 + np.exp(-np.clip(value, -20, 20)))

def softmax(xs):
    z = xs - np.max(xs)
    e = np.exp(z)
    return e / np.sum(e)

class Layer:
    def __init__(self, n_in, n_out):
        self.W = randn(n_out, n_in) / np.sqrt(n_in)
        self.b = np.zeros((n_out, 1))


class LSTM:
    def __init__(self, input_size, hidden_size):
        joined = input_size + hidden_size
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.forget = Layer(joined, hidden_size)
        self.input = Layer(joined, hidden_size)
        self.cell = Layer(joined, hidden_size)
        self.output = Layer(joined, hidden_size)
        self.forget.b[:] = 1
        self.gates = (self.forget, self.input, self.cell, self.output)

    def step(self, x, h, c):
        joined = np.vstack([x, h])
        pre = [gate.W @ joined + gate.b for gate in self.gates]
        f = sigmoid(pre[0])
        i = sigmoid(pre[1])
        g = np.tanh(pre[2])
        o = sigmoid(pre[3])
        c_new = f * c + i * g
        h_new = o * np.tanh(c_new)
        return h_new, c_new, (joined, f, i, g, o, c_new, h_new, c)


class LSTM2:
    def __init__(self, input_size, output_size, hidden_size=32):
        self.hidden_size = hidden_size
        self.lstm1 = LSTM(input_size, hidden_size)
        self.lstm2 = LSTM(hidden_size, hidden_size)
        self.layers = (self.lstm1, self.lstm2)
        self.Why = randn(output_size, hidden_size) / np.sqrt(hidden_size)
        self.by = np.zeros((output_size, 1))

    def forward(self, inputs):
        h = [np.zeros((self.hidden_size, 1)) for _ in self.layers]
        c = [np.zeros((self.hidden_size, 1)) for _ in self.layers]
        self.cache = []
        ys = []
        for x in inputs:
            step_cache = []
            current = x
            for index, layer in enumerate(self.layers):
                h[index], c[index], cached = layer.step(current, h[index], c[index])
                step_cache.append(cached)
                current = h[index]
            self.cache.append(step_cache)
            ys.append(self.Why @ h[1] + self.by)
        self.last_h = h[1]
        return ys[-1]

    def backward(self, d_y, learn_rate=0.15):
        if isinstance(d_y, list):
            d_ys = d_y
        else:
            d_ys = [np.zeros_like(d_y) for _ in self.cache]
            d_ys[-1] = d_y
        d_Why = np.zeros_like(self.Why)
        d_by = np.zeros_like(self.by)
        buckets = []
        for layer in self.layers:
            buckets.append({
                gate: {"W": np.zeros_like(gate.W), "b": np.zeros_like(gate.b)}
                for gate in layer.gates
            })
        dh_next = [np.zeros((self.hidden_size, 1)) for _ in self.layers]
        dc_next = [np.zeros((self.hidden_size, 1)) for _ in self.layers]
        for t in reversed(range(len(self.cache))):
            dh = [item.copy() for item in dh_next]
            dc = [item.copy() for item in dc_next]
            joined, f, i, g, o, c_new, h_new, c_prev = self.cache[t][1]
            grad = d_ys[t]
            d_Why += grad @ h_new.T
            d_by += grad
            dh[1] = dh[1] + self.Why.T @ grad
            for li in (1, 0):
                layer = self.layers[li]
                joined, f, i, g, o, c_new, h_new, c_prev = self.cache[t][li]
                tanh_c = np.tanh(c_new)
                d_h = dh[li]
                d_c = d_h * o * (1 - tanh_c ** 2) + dc[li]
                d_pre = [
                    (d_c * c_prev) * f * (1 - f),
                    (d_c * g) * i * (1 - i),
                    (d_c * i) * (1 - g ** 2),
                    (d_h * tanh_c) * o * (1 - o),
                ]
                d_joined = np.zeros_like(joined)
                for gate, dz in zip(layer.gates, d_pre):
                    buckets[li][gate]["W"] += dz @ joined.T
                    buckets[li][gate]["b"] += dz
                    d_joined += gate.W.T @ dz
                dh_next[li] = d_joined[layer.input_size:]
                dc_next[li] = d_c * f
                if li == 1:
                    dh[0] = dh[0] + d_joined[:layer.input_size]
        np.clip(d_Why, -5, 5, out=d_Why)
        np.clip(d_by, -5, 5, out=d_by)
        for layer, bucket in zip(self.layers, buckets):
            for gate, grad in bucket.items():
                np.clip(grad["W"], -5, 5, out=grad["W"])
                np.clip(grad["b"], -5, 5, out=grad["b"])
                gate.W -= learn_rate * grad["W"]
                gate.b -= learn_rate * grad["b"]
        self.Why -= learn_rate * d_Why
        self.by -= learn_rate * d_by


extra_train = {
    "кайф": True,
    "люблю": True,
    "огонь": True,
    "класс": True,
    "радость": True,
    "спасибо": True,
    "обожаю": True,
    "прекрасно": True,
    "мне весело": True,
    "все хорошо": True,
    "я счастлив": True,
    "не хуево": True,
    "не плохо": True,
    "вовсе не плохо": True,
    "совсем не хуево": True,
    "мне приятно": True,
    "красиво": True,
    "хорошее настроение": True,
    "ты лучший": True,
    "я рад": True,
    "не грустный": True,
    "вовсе не грустно": True,
    "бесит": False,
    "ненавижу": False,
    "отстой": False,
    "ужас": False,
    "печально": False,
    "тоска": False,
    "больно": False,
    "мерзко": False,
    "кринж": False,
    "достало": False,
    "я устал": False,
    "мне плохо": False,
    "очень плохо": False,
    "все плохо": False,
    "это ужас": False,
    "не круто": False,
    "не весело": False,
    "не хороший": False,
    "ты не хороший": False,
    "он не хороший": False,
    "совсем плохо": False,
    "плохой день": False,
    "я не рад": False,
    "очень грустно": False,
    "мне грустно очень": False,
    "это не хорошо": False,
    "вовсе не хорошо": False,
}
mood_samples = dict(train_data)
mood_samples.update(extra_train)

unk = "<unk>"
vocab = [unk] + sorted({word for text in mood_samples for word in tokens(text)})
word_to_idx = {word: index for index, word in enumerate(vocab)}
vocab_size = len(vocab)
print("vocab %d, two LSTM layers" % vocab_size)

def createInputs(text):
    words = tokens(text) or [unk]
    inputs = []
    for word in words:
        vector = np.zeros((vocab_size, 1))
        vector[word_to_idx.get(word, 0)] = 1
        inputs.append(vector)
    return inputs

np.random.seed(1)
random.seed(1)
lstm = LSTM2(vocab_size, 2)

def processData(data, backprop=True):
    items = list(data.items())
    random.shuffle(items)
    loss = 0.0
    num_correct = 0
    for text, label in items:
        target = int(label)
        out = lstm.forward(createInputs(text))
        probs = softmax(out)
        loss -= float(np.ravel(np.log(probs[target] + 1e-9))[0])
        num_correct += int(np.argmax(probs) == target)
        if backprop:
            d_y = probs.copy()
            d_y[target] -= 1
            lstm.backward(d_y)
    return loss / len(data), num_correct / len(data)

def predict_label(text):
    probs = softmax(lstm.forward(createInputs(text))).ravel()
    name = "позитив" if int(np.argmax(probs)) == 1 else "негатив"
    return name, float(probs[1])

for epoch in range(160):
    train_loss, train_acc = processData(mood_samples)
    if epoch % 40 == 39:
        test_loss, test_acc = processData(test_data, backprop=False)
        print("--- epoch %d" % (epoch + 1))
        print("Train loss %.4f | acc %.2f" % (train_loss, train_acc))
        print("Test  loss %.4f | acc %.2f" % (test_loss, test_acc))

print()
for text, label in mood_samples.items():
    name, score = predict_label(text)
    expected = "позитив" if label else "негатив"
    print("train %-22s %s -> %s %.2f" % (text, expected, name, score))
print()
for text, label in test_data.items():
    name, score = predict_label(text)
    expected = "позитив" if label else "негатив"
    print("test  %-22s %s -> %s %.2f" % (text, expected, name, score))


vocab 59, two LSTM layers
--- epoch 40
Train loss 0.4686 | acc 0.79
Test  loss 0.7164 | acc 0.57
--- epoch 80
Train loss 0.0013 | acc 1.00
Test  loss 0.0100 | acc 1.00
--- epoch 120
Train loss 0.0005 | acc 1.00
Test  loss 0.0059 | acc 1.00
--- epoch 160
Train loss 0.0003 | acc 1.00
Test  loss 0.0046 | acc 1.00

train хороший                позитив -> позитив 1.00
train круто                  позитив -> позитив 1.00
train весело                 позитив -> позитив 1.00
train ахуенно                позитив -> позитив 1.00
train ултра хорош            позитив -> позитив 1.00
train супер мега хорош       позитив -> позитив 1.00
train неплохо                позитив -> позитив 1.00
train не грустно             позитив -> позитив 1.00
train очень хорошо           позитив -> позитив 1.00
train счаслив                позитив -> позитив 1.00
train я хорош                позитив -> позитив 1.00
train я охуителен            позитив -> позитив 1.00
train мне хорошо             позитив -> позитив 1.0

На train и на test позитивная фраза остается позитивом, негативная негативом.
Фразы с `не` тоже сходятся: `я не хороший` и `мне очень грустно` уходят в негатив, `мне не грустно` остается позитивом.


Вектор настроения из двух осей.

- ось 0: `0` злой, `1` добрый
- ось 1: `0` грустный, `1` веселый

`None` значит, что эту ось на фразе не учим. `злой и грустный` это `(0, 0)`, `добрый но грустный` это `(1, 0)`, `злой но веселый` это `(0, 1)`.


In [13]:
# ось 0: 0 злой, 1 добрый
# ось 1: 0 грустный, 1 веселый
# None значит эту ось на фразе не учим
vector_train = {
    "добрый": (1, None),
    "злой": (0, None),
    "веселый": (None, 1),
    "грустный": (1, 0),
    "спокойный": (1, None),
    "раздраженный": (0, None),
    "радостный": (None, 1),
    "печальный": (1, 0),
    "круто": (1, 1),
    "радует": (1, 1),
    "ахуенно": (1, 1),
    "люблю": (1, 1),
    "кайф": (1, 1),
    "огонь": (1, 1),
    "класс": (1, 1),
    "мне хорошо": (1, 1),
    "счастье": (1, 1),
    "мне весело": (1, 1),
    "я смеюсь": (1, 1),
    "хорошее настроение": (1, 1),
    "ты лучший": (1, 1),
    "обожаю": (1, 1),
    "ура": (1, 1),
    "тоска": (1, 0),
    "печально": (1, 0),
    "мне грустно": (1, 0),
    "жалко": (1, 0),
    "слезы": (1, 0),
    "одиноко": (1, 0),
    "скучаю": (1, 0),
    "пусто на душе": (1, 0),
    "добрый но тоскливо": (1, 0),
    "хуево": (0, 0),
    "плохо": (0, 0),
    "ненавижу": (0, 0),
    "бесит": (0, 0),
    "мне плохо": (0, 0),
    "отстой": (0, 0),
    "ужас": (0, 0),
    "мерзко": (0, 0),
    "злюсь": (0, 0),
    "достало все": (0, 0),
    "плохой день": (0, 0),
    "ярость": (0, None),
    "злой и веселый": (0, 1),
    "бесит но смешно": (0, 1),
    "злорадно": (0, 1),
    "злюсь и ржу": (0, 1),
    "вредный но веселый": (0, 1),
    "добрый и веселый": (1, 1),
    "добрый и грустный": (1, 0),
    "злой и грустный": (0, 0),
    "добрый но грустный": (1, 0),
    "злой но веселый": (0, 1),
    "он добрый и смеется": (1, 1),
    "она добрая но плачет": (1, 0),
    "он злится и грустит": (0, 0),
    "он злой но шутит": (0, 1),
    "не злой": (1, None),
    "не добрый": (0, None),
    "не грустный": (None, 1),
    "не веселый": (None, 0),
    "я не злой": (1, None),
    "мне не грустно": (None, 1),
    "совсем не весело": (None, 0),
}

vector_vocab = ["<unk>"] + sorted({word for text in vector_train for word in tokens(text)})
vector_index = {word: index for index, word in enumerate(vector_vocab)}
vector_size = len(vector_vocab)

def vectorInputs(text):
    words = tokens(text) or ["<unk>"]
    inputs = []
    for word in words:
        vector = np.zeros((vector_size, 1))
        vector[vector_index.get(word, 0)] = 1
        inputs.append(vector)
    return inputs

np.random.seed(2)
random.seed(2)
vector_lstm = LSTM2(vector_size, 2)

def processVector(data, backprop=True):
    items = list(data.items())
    random.shuffle(items)
    loss = 0.0
    counted = 0
    correct = 0
    for text, target in items:
        out = vector_lstm.forward(vectorInputs(text))
        probs = sigmoid(out)
        grad = np.zeros((2, 1))
        ok = True
        used = False
        for axis, bit in enumerate(target):
            if bit is None:
                continue
            used = True
            counted += 1
            pred = float(probs[axis, 0])
            truth = float(bit)
            loss -= truth * np.log(pred + 1e-9) + (1 - truth) * np.log(1 - pred + 1e-9)
            grad[axis, 0] = pred - truth
            ok = ok and ((pred >= 0.5) == (truth >= 0.5))
        correct += int(used and ok)
        if backprop:
            vector_lstm.backward(grad)
    return loss / max(counted, 1), correct / len(data)

for epoch in range(200):
    train_loss, train_acc = processVector(vector_train)
    if epoch % 40 == 39:
        print("--- epoch %d  loss %.4f | acc %.2f" % (epoch + 1, train_loss, train_acc))

def mood_vector(text):
    kind, fun = sigmoid(vector_lstm.forward(vectorInputs(text))).ravel()
    kind_name = "добрый" if kind >= 0.5 else "злой"
    fun_name = "веселый" if fun >= 0.5 else "грустный"
    return kind_name, float(kind), fun_name, float(fun)

print()
for text in (
    "добрый",
    "злой",
    "грустный",
    "веселый",
    "круто",
    "хуево",
    "злой и грустный",
    "добрый но грустный",
    "злой но веселый",
    "ты добрый и веселый",
    "очень злой и грустный",
):
    kind_name, kind, fun_name, fun = mood_vector(text)
    print("%-24s %s %.2f | %s %.2f   vector=(%.2f, %.2f)" % (
        text, kind_name, kind, fun_name, fun, kind, fun
    ))


--- epoch 40  loss 0.1675 | acc 0.92
--- epoch 80  loss 0.0174 | acc 0.98
--- epoch 120  loss 0.0023 | acc 1.00
--- epoch 160  loss 0.0010 | acc 1.00
--- epoch 200  loss 0.0006 | acc 1.00

добрый                   добрый 1.00 | веселый 1.00   vector=(1.00, 1.00)
злой                     злой 0.00 | грустный 0.00   vector=(0.00, 0.00)
грустный                 добрый 1.00 | грустный 0.00   vector=(1.00, 0.00)
веселый                  добрый 1.00 | веселый 1.00   vector=(1.00, 1.00)
круто                    добрый 1.00 | веселый 1.00   vector=(1.00, 1.00)
хуево                    злой 0.00 | грустный 0.00   vector=(0.00, 0.00)
злой и грустный          злой 0.00 | грустный 0.00   vector=(0.00, 0.00)
добрый но грустный       добрый 1.00 | грустный 0.00   vector=(1.00, 0.00)
злой но веселый          злой 0.00 | веселый 1.00   vector=(0.00, 1.00)
ты добрый и веселый      добрый 1.00 | веселый 1.00   vector=(1.00, 1.00)
очень злой и грустный    злой 0.00 | грустный 0.00   vector=(0.00, 0.00)
